In [1]:
#prep relevent packages 
from pathlib import Path
import pandas as pd
import re

In [ ]:
# package to add to kernel to read pdfs. 
# Can also do %conda install pdfplumber if running on windows 
try:
    import pdfplumber
except ImportError:
    %conda install pdfplumber
    import pdfplumber 

In [4]:
# file path to import data
root_fldr = Path.cwd().parent
raw_fldr = root_fldr / 'data' / 'raw' 
processed_fldr = root_fldr / 'data' / 'processed'

# import cfp residential risk pdf 
commercial_risk = raw_fldr / 'cfp_data' / 'cfp_5_year_commercial_risk.pdf'

In [8]:
table_settings = {
    'vertical_strategy': 'text',
    'horizontal_strategy': 'text',
}
rows = []  # reset each run so reruns don't duplicate rows

#help provided with some AI, particularly with zip_cell check to get unified, scraped dataframe
with pdfplumber.open(commercial_risk) as pdf:
    for page in pdf.pages:
        for table in page.extract_tables(table_settings):
            for row in table:
                zip_cell = (row[0] or '').strip()
                if re.fullmatch(r'\d{5}', zip_cell):
                    # clean each cell: None -> '', drop spaces, commas, and %
                    clean_row = [(cell or '').replace(' ', '').replace(',', '').replace('%', '')
                                 for cell in row]
                    rows.append(clean_row)

#need to add 4 $ columns to align with the scraping of the pdf 
cols = ['zip',
        'growth_2025', '$', 'exposure_2025',
        'growth_2024', '$', 'exposure_2024',
        'growth_2023', '$', 'exposure_2023',
        'growth_2022', '$', 'exposure_2022',
        'growth_2021', 'exposure_2021']
commercial_risk = pd.DataFrame(rows, columns=cols)


In [9]:
#check dataframe to see if it can be read  
commercial_risk.head()

,zip,growth_2025,$,exposure_2025,growth_2024,$,exposure_2024,growth_2023,$,exposure_2023,growth_2022,$,exposure_2022,growth_2021,exposure_2021
0,94501,321,$,111564501,277,$,26509833,103,$,7035000,-7,$,3472000,$,3722000
1,94502,0,$,6430000,0,$,-,0,$,-,0,$,-,$,-
2,94536,29,$,18819000,105,$,14567000,791,$,7096000,-45,$,796000,$,1446000
3,94538,15,$,64311288,380,$,55905288,0,$,11636000,0,$,-,$,-
4,94539,57,$,4015000,0,$,2562000,34,$,2562000,0,$,1907000,$,-


In [13]:
#getting rid of $ columns 
commercial_risk = residential_risk.loc[:, residential_risk.columns != '$'] #checking all columns and mainting all those that aren't '$'
commercial_risk.head()

,zip,growth_2025,exposure_2025,growth_2024,exposure_2024,growth_2023,exposure_2023,growth_2022,exposure_2022,growth_2021,exposure_2021
0,94501,321,111564501,277,26509833,103,7035000,-7,3472000,$,3722000
1,94502,0,6430000,0,-,0,-,0,-,$,-
2,94536,29,18819000,105,14567000,791,7096000,-45,796000,$,1446000
3,94538,15,64311288,380,55905288,0,11636000,0,-,$,-
4,94539,57,4015000,0,2562000,34,2562000,0,1907000,$,-


In [14]:
#converting df into appropriate variable types 

commercial_risk_clean = (
    commercial_risk
    .apply(pd.to_numeric, errors='coerce') #converting strings to numbers, '-' becomes NaN
    .assign( #converting into percentages 
        growth_2025 = lambda d: d['growth_2025'] / 100,
        growth_2024 = lambda d: d['growth_2024'] / 100,
        growth_2023 = lambda d: d['growth_2023'] / 100,
        growth_2022 = lambda d: d['growth_2022'] / 100 
    )
    .astype({
        'growth_2025': 'float64',
        'exposure_2025': 'Int64',
        'growth_2024': 'float64',
        'exposure_2024': 'Int64',
        'growth_2023': 'float64',
        'exposure_2023': 'Int64',
        'growth_2022': 'float64',
        'exposure_2022': 'Int64',
        'exposure_2021': 'Int64'
    })
)

commercial_risk_clean.head()

,zip,growth_2025,exposure_2025,growth_2024,exposure_2024,growth_2023,exposure_2023,growth_2022,exposure_2022,growth_2021,exposure_2021
0,94501,3.21,111564501,2.77,26509833,1.03,7035000,-0.07,3472000,NaN,3722000
1,94502,0.00,6430000,0.00,<NA>,0.00,<NA>,0.00,<NA>,NaN,<NA>
2,94536,0.29,18819000,1.05,14567000,7.91,7096000,-0.45,796000,NaN,1446000
3,94538,0.15,64311288,3.80,55905288,0.00,11636000,0.00,<NA>,NaN,<NA>
4,94539,0.57,4015000,0.00,2562000,0.34,2562000,0.00,1907000,NaN,<NA>


In [15]:
commercial_risk_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1482 entries, 0 to 1481
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   zip            1482 non-null   int64  
 1   growth_2025    1482 non-null   float64
 2   exposure_2025  1429 non-null   Int64  
 3   growth_2024    1482 non-null   float64
 4   exposure_2024  1309 non-null   Int64  
 5   growth_2023    1482 non-null   float64
 6   exposure_2023  1129 non-null   Int64  
 7   growth_2022    1482 non-null   float64
 8   exposure_2022  1025 non-null   Int64  
 9   growth_2021    0 non-null      float64
 10  exposure_2021  914 non-null    Int64  
dtypes: Int64(5), float64(5), int64(1)
memory usage: 134.7 KB


In [ ]:
#check to see if numbers from pull are aligned with pdf 
# Plans from pdf
# 2025 exposure = $48,801,762,839
# 2024 exposure = $26,876,155,001
# 2023 exposure = $13,445,162,119
# 2022 exposure = $8,116,443,171

total_2025_exposure = commercial_risk_clean['exposure_2025'].sum()
total_2024_exposure = commercial_risk_clean['exposure_2024'].sum()
total_2023_exposure = commercial_risk_clean['exposure_2023'].sum()
total_2022_exposure = commercial_risk_clean['exposure_2022'].sum()

print('2025 exposure:',total_2025_exposure)
print('2024 exposure:',total_2024_exposure)
print('2023 exposure:',total_2023_exposure)
print('2023 exposure:',total_2022_exposure)



2025 exposure: 48801762839
2024 exposure: 27046604193
2023 exposure: 13445162119
2023 exposure: 8116443171


In [18]:
#converting to processed folder csv 
residential_risk_clean.to_csv(processed_fldr / 'cfp_residential_risk_cleaned.csv')
